# SAM3 → ViTPose++ Evaluation | Keypoint Detection | Stomata (4 KP)

**Detector**: SAM 3 (fine-tuned)  
**Keypoint Model**: ViTPose++ (fine-tuned)  
**Task**: Keypoint detection  
**Eval**: COCO OKS-based Keypoint AP + Bbox AP

This notebook reproduces the SAM3 → ViTPose++ evaluation reported in the CVPRW stomata keypoint benchmark.
Some public benchmark splits contain annotations only.
For those splits, users must separately download the original images from the source links listed in the Hugging Face dataset card and place them in the expected local folder structure before running evaluation.

---
## 1. Configuration & Environment

In [ ]:
import sys, os, math, json, warnings, logging, platform
import numpy as np
import cv2
import torch
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Any
from collections import defaultdict
from PIL import Image
from tqdm.notebook import tqdm

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# SAM3
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# ViTPose++
from transformers import VitPoseForPoseEstimation

warnings.filterwarnings('ignore')

print('=' * 60)
print('ENVIRONMENT')
print('=' * 60)
print(f'Python:       {sys.version.split()[0]}')
print(f'PyTorch:      {torch.__version__}')
print(f'CUDA:         {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'OpenCV:       {cv2.__version__}')
import sam3; print(f'SAM3:         {sam3.__file__}')
from importlib.metadata import version as pkg_version
print(f'pycocotools:  {pkg_version("pycocotools")}')

#============================================================
#ENVIRONMENT
#============================================================
#Python:       3.12.7
#PyTorch:      2.7.0+cu126
#CUDA:         NVIDIA H100 PCIe
#OpenCV:       4.13.0
#pycocotools:  2.0.11

In [ ]:
# ======================== CONFIGURATION ========================
# Add the dataset and model files from the hugging face links in your custom directory
# Some benchmark splits in the public dataset release contain annotations only.
# For those splits, users must separately download the original images from the source links listed in the hugging face dataset card 
# Place them in the expected folder structure before running evaluation.

from pathlib import Path
import os
import numpy as np
import torch

MODEL_NAME = "SAM3-ViTPose"
TASK = "keypoint_detection"
SAM3_BASE_WEIGHTS = Path(os.environ.get("SAM3_BASE_WEIGHTS", "weights/SAM3/base_checkpoint.pt"))
#https://huggingface.co/facebook/sam3/tree/main/sam3.pt
SAM3_FINETUNED_CKPT = Path(os.environ.get("SAM3_FINETUNED_CKPT", "weights/SAM3/checkpoint.pt"))
#https://huggingface.co/Sainath001/stomata-keypoint-benchmark-cvpr-agrivision-2026-models/tree/main/SAM3
BPE_PATH = Path(os.environ.get("BPE_PATH", "weights/SAM3/bpe_simple_vocab_16e6.txt.gz"))
#"https://github.com/facebookresearch/sam3/tree/main/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
VITPOSE_CHECKPOINT = Path(os.environ.get("VITPOSE_CHECKPOINT", "weights/ViTPose++-Huge"))
#https://huggingface.co/Sainath001/stomata-keypoint-benchmark-cvpr-agrivision-2026-models/tree/main/ViTPose%2B%2B-Huge
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "data/stomata-keypoint-benchmark/datasets"))
#https://huggingface.co/datasets/Sainath001/stomata-keypoint-benchmark-cvpr-agrivision-2026-Dataset/tree/main/datasets
SAM3_TEXT_PROMPT = "stomata"
SCORE_THRESHOLD = 0.3
MAX_DETS_PER_IMAGE = 300
STOMATA_CAT_ID = 1
NUM_KEYPOINTS = 4
OKS_SIGMAS = np.array([0.1, 0.1, 0.1, 0.1], dtype=np.float32)
IMG_H, IMG_W = 256, 192
HM_H, HM_W = 64, 48
BBOX_PAD = 1.25
MOE_DATASET_INDEX = 0
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def _has_images(d: Path) -> bool:
    exts = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp")
    for ext in exts:
        if any(d.glob(f"*{ext}")):
            return True
    return False

def build_datasets(root: str) -> dict:
    root = Path(root)
    datasets = {}
    for ds_dir in sorted(root.iterdir()):
        if not ds_dir.is_dir():
            continue
        if ds_dir.name.lower() == "all":
            continue

        gt_json = ds_dir / "coco.json"
        if not gt_json.exists():
            jsons = list(ds_dir.glob("*.json"))
            if len(jsons) == 1:
                gt_json = jsons[0]
            else:
                continue

        images_dir = ds_dir / "images"
        has_images = images_dir.is_dir() and _has_images(images_dir)

        datasets[ds_dir.name] = {
            "gt_json": str(gt_json),
            "images": str(images_dir if has_images else ds_dir),
            "has_images": has_images,
        }
    return datasets

DATASETS = build_datasets(DATA_ROOT)

RESULTS_ROOT = Path("artifacts") / MODEL_NAME
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

PRED_ROOT = RESULTS_ROOT / "preds"
PRED_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Model:           {MODEL_NAME}")
print(f"Task:            {TASK}")
print(f"Score threshold: {SCORE_THRESHOLD}")
print(f"Max dets/image:  {MAX_DETS_PER_IMAGE}")
print(f"Num keypoints:   {NUM_KEYPOINTS}")
print(f"OKS sigmas:      {OKS_SIGMAS}")
print(f"Datasets:        {list(DATASETS.keys())}")
print(f"SAM3 base:       {SAM3_BASE_WEIGHTS}")
print(f"SAM3 finetuned:  {SAM3_FINETUNED_CKPT}")
print(f"ViTPose ckpt:    {VITPOSE_CHECKPOINT}")
print(f"Pred dir:        {PRED_ROOT}")
print(f"Device:          {DEVICE}")

---
## 2. Generate COCO-format Keypoint Predictions

In [3]:
# ======================== UTILITY FUNCTIONS ========================

def xyxy_to_xywh(box_xyxy: np.ndarray) -> List[float]:
    x1, y1, x2, y2 = [float(v) for v in box_xyxy]
    return [x1, y1, max(0.0, x2 - x1), max(0.0, y2 - y1)]

def clip_xywh(x, y, w, h, W, H):
    x = max(0.0, min(float(x), W - 1.0))
    y = max(0.0, min(float(y), H - 1.0))
    w = max(0.0, min(float(w), W - x))
    h = max(0.0, min(float(h), H - y))
    return x, y, w, h

def load_coco_items(gt_json, images_root):
    coco = COCO(gt_json)
    imgs = coco.loadImgs(coco.getImgIds())
    items, size_map = [], {}
    images_root = Path(images_root)
    for im in imgs:
        iid = int(im['id'])
        H, W = int(im['height']), int(im['width'])
        fp = images_root / im['file_name']
        if not fp.exists():
            fp = images_root / Path(im['file_name']).name
        if fp.exists():
            items.append((iid, str(fp), (H, W)))
            size_map[iid] = (H, W)
    return coco, items, size_map


In [4]:
# ======================== VITPOSE++ HELPERS ========================

def _get_affine_matrix(center, scale, rot_deg, output_w, output_h):
    cx, cy = center
    sx, sy = scale
    cos_r = math.cos(math.radians(rot_deg))
    sin_r = math.sin(math.radians(rot_deg))
    rx, ry = sx / output_w, sy / output_h
    M = np.array([
        [rx * cos_r, -ry * sin_r, cx - rx * cos_r * output_w / 2 + ry * sin_r * output_h / 2],
        [rx * sin_r,  ry * cos_r, cy - rx * sin_r * output_w / 2 - ry * cos_r * output_h / 2],
    ], dtype=np.float64)
    return M

def _get_aspect_ratio_scale(bw, bh):
    target_ar = IMG_W / IMG_H
    bbox_ar = bw / max(bh, 1e-6)
    if bbox_ar > target_ar:
        scale_w = bw * BBOX_PAD
        scale_h = scale_w / target_ar
    else:
        scale_h = bh * BBOX_PAD
        scale_w = scale_h * target_ar
    return scale_w, scale_h

def decode_heatmaps(hms):
    if isinstance(hms, torch.Tensor):
        hms = hms.detach().cpu().numpy()
    K, H, W = hms.shape
    coords = np.zeros((K, 2), dtype=np.float32)
    maxvals = np.zeros((K, 1), dtype=np.float32)
    for k in range(K):
        hm = hms[k]
        idx = hm.argmax()
        y, x = divmod(int(idx), W)
        maxvals[k] = hm[y, x]
        dx = 0.25 * (hm[y, x + 1] - hm[y, x - 1]) if 0 < x < W - 1 else 0.0
        dy = 0.25 * (hm[y + 1, x] - hm[y - 1, x]) if 0 < y < H - 1 else 0.0
        coords[k] = [x + dx, y + dy]
    return coords, maxvals

def crop_and_preprocess(img_rgb, bbox_xywh):
    bx, by, bw, bh = bbox_xywh
    cx, cy = bx + bw / 2.0, by + bh / 2.0
    scale_w, scale_h = _get_aspect_ratio_scale(bw, bh)
    M = _get_affine_matrix((cx, cy), (scale_w, scale_h), 0.0, IMG_W, IMG_H)
    M_inv = cv2.invertAffineTransform(M)
    crop = cv2.warpAffine(img_rgb, M_inv, (IMG_W, IMG_H),
                          flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
    crop = (crop.astype(np.float32) / 255.0 - MEAN) / STD
    tensor = torch.from_numpy(crop.transpose(2, 0, 1)).unsqueeze(0)
    return tensor, np.array([cx, cy], dtype=np.float32), np.array([scale_w, scale_h], dtype=np.float32), M

def heatmap_to_image_coords(pred_hms, center, scale, M):
    coords_hm, maxvals = decode_heatmaps(pred_hms)
    coords_crop = np.zeros_like(coords_hm)
    coords_crop[:, 0] = coords_hm[:, 0] / (HM_W / IMG_W)
    coords_crop[:, 1] = coords_hm[:, 1] / (HM_H / IMG_H)
    ones = np.ones((coords_crop.shape[0], 1), dtype=np.float32)
    pts_h = np.hstack([coords_crop, ones])
    coords_img = (M @ pts_h.T).T
    return coords_img, maxvals


In [5]:
# ======================== SAM3 → VITPOSE++ INFERENCE ========================

def load_sam3_model(base_weights, finetuned_ckpt, bpe_path, device):
    """Load SAM3 base model + fine-tuned weights, return processor."""
    print(f'Loading SAM3 base: {base_weights}')
    model = build_sam3_image_model(
        bpe_path=bpe_path, device=device, eval_mode=True,
        enable_segmentation=True, checkpoint_path=base_weights,
    )

    print(f'Loading fine-tuned checkpoint: {finetuned_ckpt}')
    ckpt = torch.load(finetuned_ckpt, map_location="cpu", weights_only=False)
    state_dict = ckpt.get("model", ckpt.get("state_dict", ckpt))
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f'  Loaded {len(state_dict)} keys, {len(missing)} missing, {len(unexpected)} unexpected')

    model.to(device)
    model.eval()
    processor = Sam3Processor(model)
    return processor


def infer_sam3_vitpose(
    sam3_processor: Sam3Processor,
    vitpose_ckpt: str,
    items: List[Tuple[int, str, Tuple[int, int]]],
    device: str = 'cuda:0',
    score_thr: float = 0.3,
    num_kp: int = 4,
) -> Tuple[List[Dict], List[Dict]]:
    """
    Two-stage: SAM3 detects boxes → ViTPose++ predicts keypoints.
    Returns: (bbox_preds, keypoint_preds) both in COCO format.
    """
    # Load ViTPose++
    vitpose_model = VitPoseForPoseEstimation.from_pretrained(vitpose_ckpt).to(device)
    vitpose_model.eval()
    print(f'ViTPose++ loaded: {vitpose_ckpt}')

    bbox_preds = []
    kp_preds = []

    for image_id, path, (H, W) in tqdm(items, desc='SAM3→ViTPose++'):
        # --- Stage 1: SAM3 detection ---
        try:
            pil_img = Image.open(path).convert("RGB")
        except Exception:
            continue

        state = sam3_processor.set_image(pil_img)
        out = sam3_processor.set_text_prompt(state=state, prompt=SAM3_TEXT_PROMPT)

        boxes = out["boxes"]
        scores = out["scores"]

        if isinstance(boxes, torch.Tensor):
            if boxes.ndim == 3:
                boxes = boxes[0]
            boxes = boxes.detach().cpu().numpy()
        if isinstance(scores, torch.Tensor):
            if scores.ndim == 2:
                scores = scores[0]
            scores = scores.detach().cpu().numpy()

        if len(boxes) == 0:
            continue

        img_rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)

        # --- Stage 2: ViTPose++ per detection ---
        for box_xyxy, det_score in zip(boxes, scores):
            s = float(det_score)
            if s < score_thr:
                continue

            bbox_xywh = xyxy_to_xywh(box_xyxy)
            x, y, w, h = clip_xywh(*bbox_xywh, W, H)
            if w <= 0 or h <= 0:
                continue

            bbox_preds.append({
                'image_id': int(image_id),
                'category_id': STOMATA_CAT_ID,
                'bbox': [x, y, w, h],
                'score': s,
            })

            tensor, center, scale, M = crop_and_preprocess(img_rgb, [x, y, w, h])
            tensor = tensor.to(device)
            ds_idx = torch.full((1,), MOE_DATASET_INDEX, dtype=torch.long, device=device)

            with torch.no_grad():
                vp_out = vitpose_model(pixel_values=tensor, dataset_index=ds_idx)

            coords_img, maxvals = heatmap_to_image_coords(vp_out.heatmaps[0], center, scale, M)

            coco_kp = []
            for ki in range(num_kp):
                kx, ky = float(coords_img[ki, 0]), float(coords_img[ki, 1])
                v = 2 if float(maxvals[ki, 0]) > 0 else 0
                coco_kp.extend([kx, ky, v])

            kp_conf = float(np.mean(maxvals))
            kp_preds.append({
                'image_id': int(image_id),
                'category_id': STOMATA_CAT_ID,
                'bbox': [x, y, w, h],
                'score': s * kp_conf,
                'keypoints': coco_kp,
            })

    print(f'Bbox predictions: {len(bbox_preds)}, Keypoint predictions: {len(kp_preds)}')
    return bbox_preds, kp_preds


In [ ]:
# ======================== LOAD SAM3 MODEL (once) ========================

sam3_processor = load_sam3_model(
    base_weights=SAM3_BASE_WEIGHTS,
    finetuned_ckpt=SAM3_FINETUNED_CKPT,
    bpe_path=BPE_PATH,
    device=DEVICE,
)


In [ ]:
# ======================== RUN INFERENCE ON ALL SPLITS ========================

pred_files = {}
bbox_pred_files = {}

for ds_name, ds_cfg in DATASETS.items():
    print(f"\n{'=' * 60}")
    print(f'Dataset: {ds_name}')
    print(f"{'=' * 60}")

    kp_json = PRED_ROOT / f'{ds_name}_kp_preds.json'
    bbox_json = PRED_ROOT / f'{ds_name}_bbox_preds.json'

    if kp_json.exists() and bbox_json.exists():
        print(f'  Predictions already exist. Delete to regenerate.')
        pred_files[ds_name] = str(kp_json)
        bbox_pred_files[ds_name] = str(bbox_json)
        continue

    coco_gt, items, size_map = load_coco_items(ds_cfg['gt_json'], ds_cfg['images'])
    print(f'  Images found: {len(items)}')

    bbox_preds, kp_preds = infer_sam3_vitpose(
        sam3_processor=sam3_processor,
        vitpose_ckpt=VITPOSE_CHECKPOINT,
        items=items,
        device=DEVICE,
        score_thr=SCORE_THRESHOLD,
        num_kp=NUM_KEYPOINTS,
    )

    with open(kp_json, 'w') as f:
        json.dump(kp_preds, f, indent=2)
    with open(bbox_json, 'w') as f:
        json.dump(bbox_preds, f, indent=2)

    print(f'  Saved {len(kp_preds)} kp preds → {kp_json}')
    pred_files[ds_name] = str(kp_json)
    bbox_pred_files[ds_name] = str(bbox_json)

print(f"\n{'=' * 60}")
print('PREDICTION FILES:')
for name, path in pred_files.items():
    print(f'  {name}: {path}')


---
## 3. COCO Keypoint Evaluation (OKS-based AP)

In [8]:
# ======================== KEYPOINT EVALUATION ========================

def run_coco_keypoint_eval(gt_json: str, pred_json: str, sigmas: np.ndarray = None) -> Dict:
    coco_gt = COCO(gt_json)
    if 'info' not in coco_gt.dataset:
        coco_gt.dataset['info'] = {}
    coco_dt = coco_gt.loadRes(pred_json)

    coco_eval = COCOeval(coco_gt, coco_dt, iouType='keypoints')
    if sigmas is not None:
        coco_eval.params.kpt_oks_sigmas = sigmas

    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    s = coco_eval.stats
    return {
        'KP_AP': s[0], 'KP_AP50': s[1], 'KP_AP75': s[2],
        'KP_AP_M': s[3], 'KP_AP_L': s[4],
        'KP_AR': s[5], 'KP_AR50': s[6], 'KP_AR75': s[7],
        'KP_AR_M': s[8], 'KP_AR_L': s[9],
    }


In [9]:
# ======================== RUN KEYPOINT EVAL ========================

kp_results = {}

for ds_name, ds_cfg in DATASETS.items():
    print(f"\n{'=' * 60}")
    print(f'Keypoint Eval: {ds_name}')
    print(f"{'=' * 60}")

    pred_json = pred_files.get(ds_name)
    if pred_json is None or not Path(pred_json).exists():
        print(f'  Prediction file not found!')
        continue

    try:
        metrics = run_coco_keypoint_eval(
            gt_json=ds_cfg['gt_json'], pred_json=pred_json, sigmas=OKS_SIGMAS,
        )
        kp_results[ds_name] = metrics
    except Exception as e:
        print(f'  Error: {e}')
        import traceback; traceback.print_exc()

if kp_results:
    print(f"\n{'=' * 60}")
    print(f'KEYPOINT AP RESULTS — {MODEL_NAME} (OKS sigmas={OKS_SIGMAS[0]})')
    print(f"{'=' * 60}")
    df_kp = pd.DataFrame(kp_results).T
    df_kp = (df_kp * 100).round(2)
    display(df_kp)



Keypoint Eval: T_HR5MZ
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.32s).
Accumulating evaluation results...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.185
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.218
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.216
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.175
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.229
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.192
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.216
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.212
 Average Recall     (AR) @[ IoU=0.50

,KP_AP,KP_AP50,KP_AP75,KP_AP_M,KP_AP_L,KP_AR,KP_AR50,KP_AR75,KP_AR_M,KP_AR_L
T_HR5MZ,18.53,21.78,21.61,17.51,22.94,19.17,21.65,21.21,18.26,23.02
T_HR5WH,56.22,74.36,69.32,52.71,58.37,59.93,75.16,71.24,58.72,60.35
T_MSAA,27.12,43.70,27.74,27.12,-100.00,31.67,46.85,33.78,31.67,-100.00
T_MZA,37.07,51.17,43.91,37.07,-100.00,40.10,51.37,46.70,40.10,-100.00
T_MZLP,34.94,51.90,40.26,34.94,40.00,38.60,52.73,44.57,38.60,40.00
T_NKBY,29.88,53.88,25.23,-100.00,29.88,54.63,85.14,50.86,-100.00,54.63
T_SBGH,0.00,0.02,0.00,0.00,-100.00,0.06,0.38,0.00,0.15,-100.00
T_TCAB,2.28,6.59,1.04,1.00,2.51,16.04,32.84,12.69,11.67,16.48
T_TCMZ,29.42,44.08,25.16,4.58,31.42,40.37,57.41,35.93,10.50,42.76


---
## 4. COCO Bbox Evaluation

In [10]:
# ======================== BBOX EVALUATION ========================

def run_coco_bbox_eval(gt_json: str, pred_json: str) -> Dict:
    coco_gt = COCO(gt_json)
    if 'info' not in coco_gt.dataset:
        coco_gt.dataset['info'] = {}
    coco_dt = coco_gt.loadRes(pred_json)

    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    s = coco_eval.stats
    return {
        'Box_AP': s[0], 'Box_AP50': s[1], 'Box_AP75': s[2],
        'Box_AP_S': s[3], 'Box_AP_M': s[4], 'Box_AP_L': s[5],
        'Box_AR@1': s[6], 'Box_AR@10': s[7], 'Box_AR@100': s[8],
    }


bbox_results = {}

for ds_name, ds_cfg in DATASETS.items():
    print(f"\n{'=' * 60}")
    print(f'Bbox Eval: {ds_name}')
    print(f"{'=' * 60}")

    pred_json = bbox_pred_files.get(ds_name)
    if pred_json is None or not Path(pred_json).exists():
        print(f'  Prediction file not found!')
        continue

    try:
        metrics = run_coco_bbox_eval(gt_json=ds_cfg['gt_json'], pred_json=pred_json)
        bbox_results[ds_name] = metrics
    except Exception as e:
        print(f'  Error: {e}')

if bbox_results:
    print(f"\n{'=' * 60}")
    print(f'BBOX AP RESULTS — {MODEL_NAME}')
    print(f"{'=' * 60}")
    df_box = pd.DataFrame(bbox_results).T
    df_box = (df_box * 100).round(2)
    display(df_box)



Bbox Eval: T_HR5MZ
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.64s).
Accumulating evaluation results...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.504
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.908
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.494
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.491
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.573
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.008
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.072
 Average Recall     (AR) @[ IoU=0.50:0.95 | 

,Box_AP,Box_AP50,Box_AP75,Box_AP_S,Box_AP_M,Box_AP_L,Box_AR@1,Box_AR@10,Box_AR@100
T_HR5MZ,50.39,90.80,49.38,-100.0,49.08,57.31,0.77,7.15,56.76
T_HR5WH,63.00,98.60,81.12,-100.0,55.92,65.29,2.94,27.91,67.25
T_MSAA,41.55,90.61,29.45,-100.0,41.55,-100.00,1.37,15.43,50.23
T_MZA,50.96,96.63,45.01,-100.0,50.96,-100.00,1.73,16.54,58.46
T_MZLP,47.63,95.45,38.65,-100.0,47.63,10.00,1.69,16.60,55.04
T_NKBY,48.12,91.43,42.64,-100.0,-100.00,48.12,5.49,45.89,61.14
T_SBGH,0.00,0.00,0.00,0.0,0.00,-100.00,0.00,0.00,0.08
T_TCAB,49.77,77.47,57.26,-100.0,24.86,52.30,4.33,43.28,69.18
T_TCMZ,37.00,54.22,49.41,-100.0,15.99,39.11,4.63,40.59,55.74


---
## 5. Summary

In [11]:
# ======================== COMBINED SUMMARY ========================

print(f"\n{'=' * 60}")
print(f'SUMMARY — {MODEL_NAME}')
print(f'Confidence threshold: {SCORE_THRESHOLD}')
print(f'OKS sigmas: {OKS_SIGMAS.tolist()}')
print(f"{'=' * 60}")

summary_rows = []
for ds_name in DATASETS:
    row = {'Split': ds_name}
    if ds_name in kp_results:
        row.update(kp_results[ds_name])
    if ds_name in bbox_results:
        row.update(bbox_results[ds_name])
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).set_index('Split')

key_cols = ['Box_AP', 'Box_AP50', 'KP_AP', 'KP_AP50', 'KP_AP75', 'KP_AR']
key_cols = [c for c in key_cols if c in df_summary.columns]
df_key = (df_summary[key_cols] * 100).round(2)

print('\nKey Metrics (%):')
display(df_key)



SUMMARY — SAM3_ViTPose
Confidence threshold: 0.3
OKS sigmas: [0.10000000149011612, 0.10000000149011612, 0.10000000149011612, 0.10000000149011612]

Key Metrics (%):


,Box_AP,Box_AP50,KP_AP,KP_AP50,KP_AP75,KP_AR
Split,,,,,,
T_HR5MZ,50.39,90.80,18.53,21.78,21.61,19.17
T_HR5WH,63.00,98.60,56.22,74.36,69.32,59.93
T_MSAA,41.55,90.61,27.12,43.70,27.74,31.67
T_MZA,50.96,96.63,37.07,51.17,43.91,40.10
T_MZLP,47.63,95.45,34.94,51.90,40.26,38.60
T_NKBY,48.12,91.43,29.88,53.88,25.23,54.63
T_SBGH,0.00,0.00,0.00,0.02,0.00,0.06
T_TCAB,49.77,77.47,2.28,6.59,1.04,16.04
T_TCMZ,37.00,54.22,29.42,44.08,25.16,40.37


In [ ]:
# ======================== SAVE ALL RESULTS ========================

results_output = {
    'model': MODEL_NAME,
    'task': TASK,
    'score_threshold': SCORE_THRESHOLD,
    'oks_sigmas': OKS_SIGMAS.tolist(),
    'num_keypoints': NUM_KEYPOINTS,
    'keypoint_eval': kp_results,
    'bbox_eval': bbox_results,
}

results_json = PRED_ROOT / 'evaluation_results.json'
with open(results_json, 'w') as f:
    json.dump(results_output, f, indent=2, default=float)
print(f'Results saved to: {results_json}')


Expected local layout
data/stomata-keypoint-benchmark/
├── KP-Train/
├── T-MZA/
├── T-MZLP/
├── ...
weights/SAM3/
└── checkpoint.pt
weights/ViTPose++-Huge/
└── model.safetensors, config.json